In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor


EDA

In [ ]:
df_train = pd.read_csv('data/train.csv', index_col=0)
df_test = pd.read_csv('data/test.csv', index_col=0)
df_submision = pd.read_csv('data/submission_form.csv', index_col=0)

In [ ]:
df_train

In [ ]:
df_test

In [ ]:
df_submision

In [ ]:
df_train.describe(include='all')

In [ ]:
df_train.info()

In [ ]:
df_test.info()

In [ ]:
def simple_distribution_analysis(df):
    
    print("🔍 ANALYZING DATA...")
    print("=" * 50)
    
    # Step 1: Basic info
    print(f"📊 Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"📋 Column names: {list(df.columns)}")
    print()
    
    # Step 2: Find different types of columns
    number_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    text_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    print(f"🔢 Number columns: {number_cols}")
    print(f"📝 Text columns: {text_cols}")
    print()
    
    # Step 3: Check for missing values (empty cells)
    print("❓ MISSING VALUES CHECK:")
    missing = df.isnull().sum()
    missing_percent = (missing / len(df)) * 100
    
    for col in df.columns:
        if missing[col] > 0:
            print(f"   {col}: {missing[col]} missing ({missing_percent[col]:.1f}%)")
    
    if missing.sum() == 0:
        print("   ✅ No missing values! Great!")
    print()
    
    # Step 4: Analyze NUMBER columns
    if len(number_cols) > 0:
        print("🔢 NUMBER COLUMNS ANALYSIS:")
        for col in number_cols:
            print(f"\n--- {col} ---")
            data = df[col].dropna()  # Remove empty values
            
            print(f"   Count: {len(data)}")
            print(f"   Average: {data.mean():.2f}")
            print(f"   Middle value: {data.median():.2f}")
            print(f"   Lowest: {data.min():.2f}")
            print(f"   Highest: {data.max():.2f}")
            
            # Check if data is spread out or bunched together
            if data.std() > data.mean():
                print("   📈 Data is very spread out")
            else:
                print("   📊 Data is fairly consistent")
            
            # Simple outlier check
            q75 = data.quantile(0.75)
            q25 = data.quantile(0.25)
            iqr = q75 - q25
            outliers = data[(data < q25 - 1.5*iqr) | (data > q75 + 1.5*iqr)]
            
            if len(outliers) > 0:
                print(f"   ⚠️  Found {len(outliers)} unusual values (outliers)")
            else:
                print("   ✅ No unusual values")
    
    # Step 5: Analyze TEXT columns  
    if len(text_cols) > 0:
        print("\n📝 TEXT COLUMNS ANALYSIS:")
        for col in text_cols:
            print(f"\n--- {col} ---")
            data = df[col].dropna()
            
            print(f"   Count: {len(data)}")
            print(f"   Unique values: {data.nunique()}")
            
            # Show most common values
            top_values = data.value_counts().head(3)
            print("   Most common:")
            for value, count in top_values.items():
                percent = (count / len(data)) * 100
                print(f"     '{value}': {count} times ({percent:.1f}%)")
            
            # Check if data is balanced
            most_common_percent = (top_values.iloc[0] / len(data)) * 100
            if most_common_percent > 80:
                print("   ⚠️  Data is very imbalanced (one value dominates)")
            else:
                print("   ✅ Data looks balanced")

def plot_distributions(df, max_plots=6):
   
    print("\n📊 CREATING PLOTS...")
    
    # Get column types
    number_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    text_cols = df.select_dtypes(include=['object']).columns.tolist()
    
    # Plot number columns
    if len(number_cols) > 0:
        cols_to_plot = number_cols[:max_plots]
        
        fig, axes = plt.subplots(len(cols_to_plot), 2, figsize=(12, 4*len(cols_to_plot)))
        if len(cols_to_plot) == 1:
            axes = axes.reshape(1, -1)
        
        for i, col in enumerate(cols_to_plot):
            # Histogram (shows distribution shape)
            axes[i, 0].hist(df[col].dropna(), bins=30, color='skyblue', alpha=0.7)
            axes[i, 0].set_title(f'{col} - Distribution')
            axes[i, 0].set_ylabel('Count')
            
            # Box plot (shows outliers)
            axes[i, 1].boxplot(df[col].dropna())
            axes[i, 1].set_title(f'{col} - Box Plot')
            axes[i, 1].set_ylabel('Values')
        
        plt.tight_layout()
        plt.show()
    
    # Plot text columns (only if not too many categories)
    if len(text_cols) > 0:
        cols_to_plot = text_cols[:3] 
        
        for col in cols_to_plot:
            if df[col].nunique() <= 10: 
                plt.figure(figsize=(10, 6))
                
                # Count each category
                counts = df[col].value_counts().head(10)
                
                plt.bar(range(len(counts)), counts.values, color='lightcoral', alpha=0.7)
                plt.title(f'{col} - Category Counts')
                plt.xlabel('Categories')
                plt.ylabel('Count')
                plt.xticks(range(len(counts)), counts.index, rotation=45)
                
                # Add numbers on top of bars
                for i, v in enumerate(counts.values):
                    plt.text(i, v + v*0.01, str(v), ha='center')
                
                plt.tight_layout()
                plt.show()

def quick_summary(df):
    """
    Super quick overview of data
    """
    print("⚡ QUICK SUMMARY:")
    print(f"   • {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"   • {df.isnull().sum().sum()} total missing values")
    print(f"   • {len(df.select_dtypes(include=['int64', 'float64']).columns)} number columns")
    print(f"   • {len(df.select_dtypes(include=['object']).columns)} text columns")
    print()

def analyze_data(df, make_plots=False):
    
    print("🚀 STARTING DATA ANALYSIS...")
    print()
    
    # Quick overview
    quick_summary(df)
    
    # Detailed analysis
    simple_distribution_analysis(df)
    
    # Make plots
    if make_plots:
        plot_distributions(df)
    
    print("\n✅ ANALYSIS COMPLETE!")

In [ ]:
analyze_data(df_train, make_plots=False)

In [ ]:
analyze_data(df_test, make_plots=False)

Data preprocessing

In [ ]:
def drop_unnecessary_columns(df):
    
    df = df.drop(columns=['Name', 'Cabin', 'Ticket'])
    
    return df

In [ ]:
df_train = drop_unnecessary_columns(df_train)
df_test = drop_unnecessary_columns(df_test)

In [ ]:
def fill_age_missing_values(df_with_miss_values):
    
    df = df_with_miss_values.copy()
    
    df = pd.get_dummies(df)
    
    age_known = df[df['Age'].notna()]
    age_not_known = df[df['Age'].isna()]
    
    X_known = age_known.drop('Age', axis=1)
    y_known = age_known['Age']
    X_missing = age_not_known.drop('Age', axis=1)
    
    scaler = StandardScaler()
    X_known = scaler.fit_transform(X_known)
    X_missing = scaler.transform(X_missing)
    
    model = RandomForestRegressor(random_state=42)
    model.fit(X_known, y_known)
    
    predicted_ages = model.predict(X_missing)
    
    df_with_miss_values.loc[df_with_miss_values['Age'].isnull(), 'Age'] = predicted_ages
    
    return df_with_miss_values

In [ ]:
df_train = fill_age_missing_values(df_train) 
df_test = fill_age_missing_values(df_test)

In [ ]:
def filling_other_miss_values(df):

    columns_with_na = df.columns[df.isnull().any()].tolist()
    
    for column in columns_with_na:
        
        if df[column].dtype != 'object':
            
            df[column] = df[column].fillna(df[column].median())
            
        else:
            
            df[column] = df[column].fillna(df[column].mode()[0])
    
    return df

In [ ]:
df_train = filling_other_miss_values(df_train)
df_test = filling_other_miss_values(df_test)

In [ ]:
sns.histplot(df_train['Age'])
df_train['Age'].skew()

In [ ]:
sns.histplot(df_train['SibSp'])
df_train['SibSp'].skew()

In [ ]:
sns.histplot(df_train['Parch'])
df_train['Parch'].skew()

In [ ]:
sns.histplot(df_train['Fare'])
df_train['Fare'].skew()

In [ ]:
def fare_encoding(df_train, df_test):
    
    df_train['Fare_log'] = np.log1p(df_train['Fare'])
    df_test['Fare_log'] = np.log1p(df_test['Fare'])
    
    quantiles = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
    bin_edges = df_train['Fare_log'].quantile(quantiles).values

    df_train['Fare_log_binned'] = pd.cut(df_train['Fare_log'], 
                                        bins=bin_edges, 
                                        labels=['Very_Low', 'Low', 'Medium', 'High', 'Very_High'],
                                        include_lowest=True)
    df_test['Fare_log_binned'] = pd.cut(df_test['Fare_log'], 
                                       bins=bin_edges, 
                                       labels=['Very_Low', 'Low', 'Medium', 'High', 'Very_High'],
                                       include_lowest=True)
    
    fare_encoding = df_train.groupby('Fare_log_binned')['Survived'].mean()
    
    df_train['Fare_encoded'] = df_train['Fare_log_binned'].map(fare_encoding).astype(float)
    df_test['Fare_encoded'] = df_test['Fare_log_binned'].map(fare_encoding).astype(float)
    
    df_train = df_train.drop(columns=['Fare', 'Fare_log', 'Fare_log_binned'])
    df_test = df_test.drop(columns=['Fare', 'Fare_log', 'Fare_log_binned'])
    
    return df_train, df_test


In [ ]:
df_train, df_test = fare_encoding(df_train, df_test)

In [ ]:
# Count and survival rate together
survival_by_port = df_train.groupby('Embarked')['Survived'].agg(['count', 'mean'])
print(survival_by_port)

# Add percentage for easier reading
survival_by_port['survival_pct'] = (survival_by_port['mean'] * 100).round(1)
print(survival_by_port)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.barplot(data=df_train, x='Embarked', y='Survived')
plt.title('Survival Rate by Embarkation Port')
plt.ylabel('Survival Rate')
plt.show()

In [ ]:
def embarked_encoding(df_train, df_test):
    
    embarked_encoding = df_train.groupby('Embarked')['Survived'].mean()
    
    df_train['Embarked_encoded'] = df_train['Embarked'].map(embarked_encoding).astype(float)
    df_test['Embarked_encoded'] = df_test['Embarked'].map(embarked_encoding).astype(float)
    
    df_train = df_train.drop(columns=['Embarked'])
    df_test = df_test.drop(columns=['Embarked'])
    
    return df_train, df_test


In [ ]:
df_train, df_test = embarked_encoding(df_train, df_test)

In [ ]:
def sex_encoding(df_train, df_test):
    
    df_train['IsMale'] = (df_train['Sex'] == 'male').astype(int)
    df_test['IsMale'] = (df_test['Sex'] == 'male').astype(int)
    
    df_train = df_train.drop(columns=['Sex'])
    df_test = df_test.drop(columns=['Sex'])
    
    return df_train, df_test

In [ ]:
df_train, df_test = sex_encoding(df_train, df_test)

In [ ]:
analyze_data(df_train)

In [ ]:
df_train.info()

Feature Engineering